# 🛒 K-Means Clustering on Konga E-Commerce Data
## Nigerian AI & Data Science Community — Data Science Bootcamp · Session 7

**Objective:** Walk through K-Means clustering step-by-step using a simulated Konga transaction dataset to segment customers by purchasing behaviour.

**What you'll learn:**
1. Feature engineering from raw transactions
2. Data preprocessing for clustering
3. Choosing K — Elbow Method & Silhouette Analysis
4. Fitting and interpreting K-Means clusters
5. Visualising cluster formation
6. Evaluating with Davies-Bouldin Index
7. Applying MiniBatch K-Means for scale

---
> 📁 **Dataset:** `konga_transactions.csv` — 5,024 rows, 1,813 orders, Jan–Dec 2024 (simulated)


## 0. 📦 Install & Import Libraries

In [ ]:
# All standard libraries — no extra installs needed in a typical Python DS environment
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, davies_bouldin_score
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings("ignore")

# Plot aesthetics
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F5F9FF",
    "axes.grid":        True,
    "grid.color":       "#D0E4F7",
    "grid.linewidth":   0.6,
    "font.family":      "DejaVu Sans",
})
PALETTE = ["#1A3A5C", "#2196F3", "#4CAF50", "#FF9800", "#E91E63", "#9C27B0"]

print("✅  Libraries loaded successfully")


---
## 1. 📂 Load & Explore the Dataset

We start by loading the raw Konga transaction log. Each row represents one **line item** (one product in one order).


In [ ]:
df = pd.read_csv("konga_transactions.csv", parse_dates=["order_date"])

print(f"Shape: {df.shape}")
print(f"Unique orders:    {df['order_id'].nunique():,}")
print(f"Unique customers: {df['customer_id'].nunique():,}")
print(f"Date range:       {df['order_date'].min().date()}  →  {df['order_date'].max().date()}")
df.head(8)


In [ ]:
# Quick data health check
print("=== Data Types ===")
print(df.dtypes)
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Numeric Summary ===")
df[["quantity", "unit_price_ngn", "line_total_ngn"]].describe()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Konga Dataset — Exploratory Distributions", fontsize=14, fontweight="bold", y=1.02)

# Orders by city
city_counts = df.groupby("city")["order_id"].nunique().sort_values(ascending=True)
city_counts.plot(kind="barh", ax=axes[0], color=PALETTE[0])
axes[0].set_title("Orders by City"); axes[0].set_xlabel("Unique Orders")

# Revenue by category
cat_rev = df.groupby("category")["line_total_ngn"].sum().sort_values() / 1e6
cat_rev.plot(kind="barh", ax=axes[1], color=PALETTE[1])
axes[1].set_title("Revenue by Category (₦M)"); axes[1].set_xlabel("₦ Million")

# Payment method share
pay_counts = df["payment_method"].value_counts()
axes[2].pie(pay_counts, labels=pay_counts.index, autopct="%1.0f%%",
            colors=PALETTE, startangle=90, pctdistance=0.8)
axes[2].set_title("Payment Method Share")

plt.tight_layout()
plt.show()


---
## 2. 🔧 Feature Engineering — Building a Customer Profile

K-Means works on a **feature matrix** — one row per customer.  
We'll engineer **RFM-inspired features** from the raw transactions:

| Feature | Description |
|---|---|
| `total_orders` | How many distinct orders the customer placed |
| `total_spend_ngn` | Total ₦ spent across all orders |
| `avg_order_value` | Mean spend per order |
| `total_items` | Total quantity of products bought |
| `unique_categories` | Breadth of shopping — how many categories explored |
| `favourite_category` | Most purchased category (encoded numerically) |
| `avg_unit_price` | Average price of items purchased (proxy for price sensitivity) |
| `orders_per_month` | Purchase frequency |

> 💡 **Why RFM?** Recency, Frequency, and Monetary value are the gold standard for e-commerce segmentation.


In [ ]:
# Step 1 — Order-level aggregation
order_agg = (
    df.groupby(["customer_id", "order_id"])
      .agg(
          order_value   = ("line_total_ngn", "sum"),
          items_in_cart = ("quantity", "sum"),
          order_date    = ("order_date", "first"),
      )
      .reset_index()
)

# Step 2 — Customer-level aggregation
customer_features = (
    order_agg.groupby("customer_id")
             .agg(
                 total_orders      = ("order_id",    "nunique"),
                 total_spend_ngn   = ("order_value", "sum"),
                 avg_order_value   = ("order_value", "mean"),
                 total_items       = ("items_in_cart","sum"),
                 first_order       = ("order_date",  "min"),
                 last_order        = ("order_date",  "max"),
             )
             .reset_index()
)

# Step 3 — Category diversity per customer
cat_diversity = (
    df.groupby("customer_id")
      .agg(
          unique_categories = ("category", "nunique"),
          avg_unit_price    = ("unit_price_ngn", "mean"),
      )
      .reset_index()
)

# Step 4 — Purchase frequency (orders per month active)
customer_features["months_active"] = (
    (customer_features["last_order"] - customer_features["first_order"])
    .dt.days / 30
).clip(lower=1)
customer_features["orders_per_month"] = (
    customer_features["total_orders"] / customer_features["months_active"]
)

# Step 5 — Merge everything
customers = customer_features.merge(cat_diversity, on="customer_id")
customers = customers.drop(columns=["first_order", "last_order", "months_active"])

print(f"Customer feature matrix: {customers.shape}")
customers.head()


In [ ]:
# Feature distributions
features = ["total_orders", "total_spend_ngn", "avg_order_value",
            "total_items", "unique_categories", "avg_unit_price", "orders_per_month"]

fig, axes = plt.subplots(2, 4, figsize=(18, 7))
fig.suptitle("Customer Feature Distributions (before scaling)", fontsize=14, fontweight="bold")
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].hist(customers[feat], bins=30, color=PALETTE[i % len(PALETTE)], edgecolor="white")
    axes[i].set_title(feat, fontsize=10)
    axes[i].set_xlabel("Value")
    axes[i].set_ylabel("Count")

axes[-1].set_visible(False)
plt.tight_layout()
plt.show()

print("\n⚠️  Notice the right-skewed distributions — StandardScaler will normalise these for K-Means.")


---
## 3. ⚖️ Preprocessing — Why Scaling Matters for K-Means

K-Means relies on **Euclidean distance**. If `total_spend_ngn` ranges in the millions while `unique_categories` only goes 1–10, spend will completely dominate the distance calculation — and K-Means will essentially only cluster on spend.

**StandardScaler** transforms each feature to **mean=0, std=1**, giving every feature equal weight.


In [ ]:
X_raw = customers[features].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X_scaled_df = pd.DataFrame(X_scaled, columns=features, index=customers["customer_id"])

# Show the effect of scaling
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Before
axes[0].boxplot([X_raw[f] for f in features], labels=features, vert=False)
axes[0].set_title("BEFORE Scaling — Raw Feature Ranges", fontweight="bold")
axes[0].set_xlabel("Raw Value")

# After
axes[1].boxplot([X_scaled_df[f] for f in features], labels=features, vert=False)
axes[1].set_title("AFTER Scaling — Normalised (mean=0, std=1)", fontweight="bold")
axes[1].set_xlabel("Scaled Value")
axes[1].axvline(0, color="red", linestyle="--", linewidth=1, alpha=0.6)

plt.tight_layout()
plt.show()

print(f"✅  Feature matrix ready: {X_scaled.shape[0]} customers × {X_scaled.shape[1]} features")


---
## 4. 🔍 Choosing K — Elbow Method + Silhouette Analysis

We **never guess K**. We run K-Means for a range of K values and examine two metrics:

- **Inertia (WCSS)**: Sum of squared distances of each point to its centroid. Lower = tighter clusters. We look for the "elbow".
- **Silhouette Score**: Measures how similar a point is to its own cluster vs neighbouring clusters. Ranges –1 to +1. Higher = better.


In [ ]:
K_range = range(2, 11)
inertias        = []
silhouette_scores = []

print("Computing K-Means for K = 2 to 10...")
for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouette_scores.append(sil)
    print(f"  K={k}  |  Inertia={km.inertia_:>10,.1f}  |  Silhouette={sil:.4f}")

print("\n✅  Done!")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Choosing the Optimal K", fontsize=14, fontweight="bold")

k_list = list(K_range)

# ── Elbow plot ──
ax1.plot(k_list, inertias, "o-", color=PALETTE[0], linewidth=2.5, markersize=8)
ax1.fill_between(k_list, inertias, alpha=0.1, color=PALETTE[0])
ax1.set_title("Elbow Method — Inertia (WCSS) vs K", fontweight="bold")
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("Inertia (WCSS)")

# Mark elbow (biggest second-derivative drop)
diffs2 = np.diff(np.diff(inertias))
elbow_k = k_list[np.argmax(diffs2) + 1]
ax1.axvline(elbow_k, color="red", linestyle="--", linewidth=1.8, label=f"Elbow at K={elbow_k}")
ax1.annotate(f"  Elbow\n  K={elbow_k}", xy=(elbow_k, inertias[elbow_k-2]),
             fontsize=11, color="red", fontweight="bold")
ax1.legend()

# ── Silhouette plot ──
best_sil_k = k_list[np.argmax(silhouette_scores)]
bar_colors = [PALETTE[0] if k != best_sil_k else "#E91E63" for k in k_list]
bars = ax2.bar(k_list, silhouette_scores, color=bar_colors, edgecolor="white", width=0.6)
ax2.set_title("Silhouette Score vs K  (higher = better)", fontweight="bold")
ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Avg Silhouette Score")
ax2.set_ylim(0, max(silhouette_scores) * 1.2)

for bar, score in zip(bars, silhouette_scores):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f"{score:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

ax2.axhline(silhouette_scores[best_sil_k - 2], color="#E91E63",
            linestyle="--", linewidth=1.5, label=f"Best K={best_sil_k}")
ax2.legend()

plt.tight_layout()
plt.show()

print(f"\n📌  Elbow suggests K={elbow_k}")
print(f"📌  Silhouette score peaks at K={best_sil_k}")
print(f"\n👉  We'll proceed with K={best_sil_k} (confirmed by both methods).")
OPTIMAL_K = best_sil_k


---
## 5. 📊 Silhouette Plot — Individual Customer Analysis

A silhouette plot shows each customer's silhouette score as a horizontal bar. Wide, uniform bars = well-separated, tight clusters. Thin or negative bars = problematic assignments.


In [ ]:
from sklearn.metrics import silhouette_samples
import matplotlib.cm as cm

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Silhouette Plots for K = 3, 4, 5", fontsize=14, fontweight="bold")

for idx, k in enumerate([3, 4, 5]):
    ax = axes[idx]
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    sil_vals = silhouette_samples(X_scaled, labels)
    avg_sil  = silhouette_score(X_scaled, labels)

    y_lower = 10
    cmap = cm.get_cmap("tab10")
    for cluster in range(k):
        cluster_vals = np.sort(sil_vals[labels == cluster])
        size = cluster_vals.shape[0]
        y_upper = y_lower + size
        color = cmap(cluster / k)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_vals,
                         facecolor=color, edgecolor=color, alpha=0.8)
        ax.text(-0.05, y_lower + size / 2, f"C{cluster}", fontsize=9)
        y_lower = y_upper + 10

    ax.axvline(avg_sil, color="red", linestyle="--", linewidth=1.5)
    ax.set_title(f"K={k}  |  Avg Silhouette = {avg_sil:.3f}", fontweight="bold")
    ax.set_xlabel("Silhouette Coefficient")
    ax.set_ylabel("Cluster")
    ax.set_xlim(-0.2, 1.0)
    ax.set_yticks([])

plt.tight_layout()
plt.show()

print("\n💡  Look for: wide bars above the red dashed line — that's a well-formed cluster.")
print("    Bars dipping below 0 mean those customers are likely misassigned.")


---
## 6. ⚙️ Fitting the Final K-Means Model (K = Optimal)

Now we fit our final model using:
- **K-Means++ initialisation** — smarter centroid placement, avoids bad random starts
- **n_init=10** — runs 10 times with different seeds, keeps the best
- **max_iter=300** — convergence limit


In [ ]:
kmeans_final = KMeans(
    n_clusters = OPTIMAL_K,
    init       = "k-means++",
    n_init     = 10,
    max_iter   = 300,
    random_state = 42,
    verbose    = 0,
)

kmeans_final.fit(X_scaled)
customers["cluster"] = kmeans_final.labels_

print(f"✅  K-Means converged in {kmeans_final.n_iter_} iterations")
print(f"    Final Inertia (WCSS): {kmeans_final.inertia_:,.1f}")
print(f"\nCluster sizes:")
print(customers["cluster"].value_counts().sort_index().to_string())


---
## 7. 🎬 Visualising Cluster Formation (Convergence Steps)

To understand *how* K-Means works, we reduce features to 2D with PCA and manually step through early iterations.


In [ ]:
# Reduce to 2D for visualisation
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_

print(f"PCA explains {explained[0]*100:.1f}% + {explained[1]*100:.1f}% = {sum(explained)*100:.1f}% of variance")

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("K-Means Convergence — Iteration Steps (PCA 2D Projection)", fontsize=14, fontweight="bold")

step_labels = ["Iteration 1", "Iteration 2", "Iteration 5", f"Converged (iter {kmeans_final.n_iter_})"]
iter_counts  = [1, 2, 5, kmeans_final.n_iter_]

cmap_pts = [PALETTE[i] for i in range(OPTIMAL_K)]

for ax, n_iter, title in zip(axes, iter_counts, step_labels):
    km_step = KMeans(
        n_clusters=OPTIMAL_K, init="k-means++", n_init=1,
        max_iter=n_iter, random_state=42
    )
    step_labels_arr = km_step.fit_predict(X_scaled)
    centroids_2d = pca.transform(km_step.cluster_centers_)

    for c in range(OPTIMAL_K):
        mask = step_labels_arr == c
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=cmap_pts[c], alpha=0.4, s=18, label=f"C{c}")

    ax.scatter(centroids_2d[:, 0], centroids_2d[:, 1],
               c="black", marker="*", s=220, zorder=5, label="Centroid")

    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_xlabel(f"PC1 ({explained[0]*100:.0f}%)")
    ax.set_ylabel(f"PC2 ({explained[1]*100:.0f}%)")

axes[-1].legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()


---
## 8. 🏷️ Cluster Interpretation — Who is in Each Segment?

The mathematical clusters are only useful if we can attach **business meaning** to them.  
We examine the mean feature values per cluster and give each segment a name.


In [ ]:
# Mean feature values per cluster (in original scale for readability)
cluster_profile = customers.groupby("cluster")[features].mean()
cluster_profile["n_customers"] = customers.groupby("cluster").size()
cluster_profile = cluster_profile.round(2)
print("=== Cluster Profiles (original scale) ===")
cluster_profile


In [ ]:
# Heatmap of normalised cluster means — shows what makes each cluster unique
cluster_means_scaled = pd.DataFrame(
    scaler.transform(cluster_profile[features]),
    columns=features,
    index=cluster_profile.index
)

fig, ax = plt.subplots(figsize=(12, max(4, OPTIMAL_K * 1.2)))
sns.heatmap(
    cluster_means_scaled,
    annot=True, fmt=".2f",
    cmap="RdYlGn",
    center=0,
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Standard Deviations from Mean"},
)
ax.set_title("Cluster Feature Heatmap\n(green = above average, red = below average)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Feature")
ax.set_ylabel("Cluster")
plt.tight_layout()
plt.show()

print("\n💡 How to read this:")
print("   Green  = that cluster is ABOVE average on this feature")
print("   Red    = that cluster is BELOW average on this feature")


In [ ]:
# Radar / spider chart — one polygon per cluster
from matplotlib.patches import FancyArrowPatch

angles = np.linspace(0, 2 * np.pi, len(features), endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
ax.set_facecolor("#F5F9FF")

for i, row in cluster_means_scaled.iterrows():
    values = row.tolist() + row.tolist()[:1]
    ax.plot(angles, values, "o-", linewidth=2, color=PALETTE[i], label=f"Cluster {i}")
    ax.fill(angles, values, alpha=0.12, color=PALETTE[i])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.replace("_", "\n") for f in features], size=9)
ax.set_title("Cluster Radar Chart\n(normalised feature means)", fontsize=13,
             fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()


---
## 9. 🏆 Naming the Customer Segments

Based on the heatmap and radar chart, assign meaningful names to each cluster.  
*(Names will adapt to what K your data produces — interpret based on the heatmap above.)*


In [ ]:
# Automated segment labelling based on feature dominance
def label_cluster(row):
    if row["total_spend_ngn"] > cluster_profile["total_spend_ngn"].median() * 1.5:
        if row["unique_categories"] >= cluster_profile["unique_categories"].median():
            return "⭐ Premium Multi-Category Shopper"
        return "💎 High-Value Focused Buyer"
    elif row["total_orders"] > cluster_profile["total_orders"].median() * 1.2:
        return "🔁 Frequent Budget Shopper"
    elif row["avg_unit_price"] < cluster_profile["avg_unit_price"].median():
        return "💰 Price-Sensitive Bargain Hunter"
    elif row["unique_categories"] >= cluster_profile["unique_categories"].median():
        return "🛍️ Casual Explorer"
    else:
        return "😴 Dormant / One-Time Buyer"

cluster_profile["segment_name"] = cluster_profile.apply(label_cluster, axis=1)
customers = customers.merge(
    cluster_profile[["segment_name"]].reset_index(),
    on="cluster"
)

print("=== Final Segment Assignments ===")
print(cluster_profile[["n_customers", "total_spend_ngn", "avg_order_value",
                         "unique_categories", "orders_per_month", "segment_name"]].to_string())


In [ ]:
# Pie chart of segment sizes
seg_counts = customers["segment_name"].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Customer Segment Distribution — Konga 2024", fontsize=14, fontweight="bold")

wedges, texts, autotexts = ax1.pie(
    seg_counts,
    labels=seg_counts.index,
    autopct="%1.1f%%",
    colors=PALETTE[:len(seg_counts)],
    startangle=90,
    pctdistance=0.82,
    wedgeprops=dict(width=0.55),
)
for t in autotexts: t.set_fontsize(10); t.set_fontweight("bold")
ax1.set_title("Proportion of Customers per Segment")

# Avg spend per segment
avg_spend = customers.groupby("segment_name")["total_spend_ngn"].mean().sort_values()
bars = ax2.barh(avg_spend.index, avg_spend.values / 1000,
                color=PALETTE[:len(avg_spend)], edgecolor="white")
for bar, val in zip(bars, avg_spend.values):
    ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
             f"₦{val/1000:,.0f}K", va="center", fontsize=9, fontweight="bold")
ax2.set_title("Average Total Spend per Segment (₦'000)")
ax2.set_xlabel("₦ Thousands")

plt.tight_layout()
plt.show()


---
## 10. 🗺️ Final Cluster Visualisation (PCA 2D)

Plot all customers in 2D (PCA-reduced) space, coloured by their final cluster assignment.


In [ ]:
final_labels = customers["cluster"].values
centroids_2d = pca.transform(kmeans_final.cluster_centers_)

fig, ax = plt.subplots(figsize=(12, 8))
ax.set_facecolor("#F0F6FF")

for c in range(OPTIMAL_K):
    mask = final_labels == c
    seg  = cluster_profile.loc[c, "segment_name"]
    ax.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        c=PALETTE[c], alpha=0.55, s=25,
        label=f"C{c}: {seg}",
        edgecolors="white", linewidths=0.3,
    )

# Centroids
ax.scatter(
    centroids_2d[:, 0], centroids_2d[:, 1],
    c="black", marker="*", s=350, zorder=6, label="Centroids",
    edgecolors="white", linewidths=1,
)

for i, (x, y) in enumerate(centroids_2d):
    ax.annotate(f"C{i}", (x, y), textcoords="offset points",
                xytext=(8, 8), fontsize=11, fontweight="bold", color="black")

ax.set_title("K-Means Customer Segmentation — Konga 2024\n(PCA 2D Projection)",
             fontsize=14, fontweight="bold")
ax.set_xlabel(f"Principal Component 1  ({explained[0]*100:.1f}% variance)")
ax.set_ylabel(f"Principal Component 2  ({explained[1]*100:.1f}% variance)")
ax.legend(loc="upper right", fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()


---
## 11. 📏 Clustering Evaluation Metrics

We evaluate quality using three complementary metrics:

| Metric | Formula | Goal |
|---|---|---|
| **Silhouette Score** | `(b-a) / max(a,b)` | Closer to **+1** = better |
| **Davies-Bouldin Index** | `mean(max(σᵢ+σⱼ)/d(cᵢ,cⱼ))` | Closer to **0** = better |
| **Inertia (WCSS)** | `Σ‖xᵢ - μₖ‖²` | Lower = tighter clusters |


In [ ]:
final_labels_arr = kmeans_final.labels_

sil   = silhouette_score(X_scaled, final_labels_arr)
db    = davies_bouldin_score(X_scaled, final_labels_arr)
wcss  = kmeans_final.inertia_

print("=" * 45)
print(f"  K-Means Evaluation  (K = {OPTIMAL_K})")
print("=" * 45)
print(f"  Silhouette Score       : {sil:.4f}   {'✅ Good' if sil > 0.4 else '⚠️  Moderate'}")
print(f"  Davies-Bouldin Index   : {db:.4f}   {'✅ Good' if db < 1.5 else '⚠️  Moderate'}")
print(f"  Inertia (WCSS)         : {wcss:,.1f}")
print("=" * 45)
print()

# Comparison table across K values
print("=== Full Comparison Table ===")
print(f"{'K':>3} | {'Inertia':>12} | {'Silhouette ↑':>14} | {'Davies-Bouldin ↓':>17} | Verdict")
print("-" * 65)
for k in range(2, 11):
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=42)
    lbl = km.fit_predict(X_scaled)
    s = silhouette_score(X_scaled, lbl)
    d = davies_bouldin_score(X_scaled, lbl)
    verdict = " ⭐ OPTIMAL" if k == OPTIMAL_K else ""
    print(f"{k:>3} | {km.inertia_:>12,.1f} | {s:>14.4f} | {d:>17.4f} |{verdict}")


---
## 12. ⚡ MiniBatch K-Means — Scaling to Millions of Records

Standard K-Means processes **all data** every iteration → slow at scale.  
MiniBatch K-Means uses **random mini-batches** → ~3–10× faster with minimal quality loss.

> 🇳🇬 **Nigerian context:** MTN has 70M+ subscribers. Processing all records with standard K-Means could take hours. MiniBatch reduces this to minutes.


In [ ]:
import time

# Standard K-Means timing
t0 = time.time()
km_standard = KMeans(n_clusters=OPTIMAL_K, init="k-means++", n_init=10, random_state=42)
labels_std = km_standard.fit_predict(X_scaled)
t_standard = time.time() - t0

# MiniBatch K-Means timing
t0 = time.time()
km_mini = MiniBatchKMeans(n_clusters=OPTIMAL_K, batch_size=100,
                           init="k-means++", n_init=10, random_state=42)
labels_mini = km_mini.fit_predict(X_scaled)
t_mini = time.time() - t0

sil_std  = silhouette_score(X_scaled, labels_std)
sil_mini = silhouette_score(X_scaled, labels_mini)

print("=" * 52)
print(f"  {'':20s} {'Standard':>12}  {'MiniBatch':>12}")
print("=" * 52)
print(f"  {'Time (seconds)':20s} {t_standard:>12.4f}  {t_mini:>12.4f}")
print(f"  {'Inertia':20s} {km_standard.inertia_:>12,.1f}  {km_mini.inertia_:>12,.1f}")
print(f"  {'Silhouette Score':20s} {sil_std:>12.4f}  {sil_mini:>12.4f}")
print(f"  {'Speedup':20s} {'':>12}  {t_standard/max(t_mini,0.0001):>11.1f}×")
print("=" * 52)
print("\n💡 Quality difference is minimal — MiniBatch is preferred for large datasets.")


In [ ]:
# Side-by-side cluster comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Standard K-Means vs MiniBatch K-Means\n(PCA 2D Projection)", fontsize=14, fontweight="bold")

for ax, labels, title in [(ax1, labels_std, "Standard K-Means"), (ax2, labels_mini, "MiniBatch K-Means")]:
    for c in range(OPTIMAL_K):
        mask = labels == c
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=PALETTE[c], alpha=0.5, s=20, label=f"C{c}")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


---
## 13. 💼 Business Recommendations by Segment

The clusters only create value if translated into **actionable strategies**.


In [ ]:
recommendations = {
    "⭐ Premium Multi-Category Shopper": {
        "strategy":  "Exclusive loyalty programme, early access to new products",
        "channel":   "Personalised email + Konga app push notifications",
        "offer":     "VIP membership with free delivery + cashback rewards",
        "priority":  "🔴 HIGH — protect this segment at all costs",
    },
    "💎 High-Value Focused Buyer": {
        "strategy":  "Cross-sell into adjacent categories they haven't explored",
        "channel":   "In-app targeted banners when browsing their core category",
        "offer":     "Bundle deals — e.g. phone + accessories at 10% off",
        "priority":  "🟠 HIGH — expand their basket",
    },
    "🔁 Frequent Budget Shopper": {
        "strategy":  "Subscription-style repeat purchase incentives",
        "channel":   "SMS + USSD promotions (price-sensitive — direct)",
        "offer":     "Buy 3 get 1 free on FMCG/grocery categories",
        "priority":  "🟡 MEDIUM — increase average order value",
    },
    "💰 Price-Sensitive Bargain Hunter": {
        "strategy":  "Flash sales, countdown deals, low-price guarantees",
        "channel":   "WhatsApp broadcast deals, Pay on Delivery option",
        "offer":     "Daily deals page, Konga Express budget tier",
        "priority":  "🟡 MEDIUM — convert with urgency",
    },
    "🛍️ Casual Explorer": {
        "strategy":  "Discover & browse nudges, curated collections",
        "channel":   "Instagram/social retargeting, wishlist reminders",
        "offer":     "Free shipping on first purchase this month",
        "priority":  "🟢 MEDIUM — increase engagement",
    },
    "😴 Dormant / One-Time Buyer": {
        "strategy":  "Win-back campaign — remind them of Konga",
        "channel":   "Email re-engagement sequence (3 emails over 2 weeks)",
        "offer":     "₦1,000 voucher valid 7 days — create urgency",
        "priority":  "🔵 LOW — re-activate cheaply or accept churn",
    },
}

print("\n╔══════════════════════════════════════════════════════════════╗")
print("║         KONGA CUSTOMER SEGMENT PLAYBOOK                     ║")
print("╚══════════════════════════════════════════════════════════════╝")

seg_names = cluster_profile["segment_name"].tolist()
for seg in seg_names:
    if seg in recommendations:
        r = recommendations[seg]
        print(f"\n{'='*66}")
        print(f" {seg}")
        print(f"{'='*66}")
        print(f"  Strategy  : {r['strategy']}")
        print(f"  Channel   : {r['channel']}")
        print(f"  Offer     : {r['offer']}")
        print(f"  Priority  : {r['priority']}")


---
## 14. ✅ Summary & Next Steps

### What we did in this notebook:

| Step | Action |
|---|---|
| 1 | Loaded raw Konga transaction data (5,024 rows) |
| 2 | Engineered RFM-inspired customer features |
| 3 | Scaled features with StandardScaler |
| 4 | Used Elbow + Silhouette methods to choose optimal K |
| 5 | Fitted K-Means with K-Means++ initialisation |
| 6 | Visualised convergence steps via PCA |
| 7 | Evaluated with Silhouette, Davies-Bouldin, and Inertia |
| 8 | Applied MiniBatch K-Means for large-scale comparison |
| 9 | Named segments and produced business recommendations |

### 🚀 Your Project Checklist:
- [ ] Select your own domain dataset (Kaggle, UCI, data.gov.ng)
- [ ] Engineer meaningful features for your domain  
- [ ] Preprocess: handle nulls, scale features  
- [ ] Run K-Means for K=2–10, plot elbow + silhouette  
- [ ] Fit final model, name clusters  
- [ ] Write 3+ business recommendations per segment  
- [ ] Submit: notebook + 5-min walkthrough video  

### 📚 Further Reading:
- [scikit-learn KMeans docs](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)  
- [Silhouette Analysis Guide](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html)
- [Kaggle: Customer Segmentation notebook](https://www.kaggle.com/code/fabiendaniel/customer-segmentation)
